# Hey Sense Intelligence Engine

## Fase 1: Data Cleaning + Data Validation

Objetivo: preparar las fuentes de conversaciones, clientes, productos y transacciones para construir una vista **Customer 360** por `user_id`.

Pipeline general:

```text
Raw Data -> Data Cleaning -> Data Validation -> Feature Engineering -> Customer 360 -> Model Layer -> Need Detection -> Next Best Action
```

## 1. Imports y rutas

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

BASE_DIR = Path.cwd()

CONV_PATH = BASE_DIR / "dataset_conversaciones" / "dataset_50k_anonymized.csv"
CLIENTES_PATH = BASE_DIR / "dataset_transacciones" / "hey_clientes.csv"
PRODUCTOS_PATH = BASE_DIR / "dataset_transacciones" / "hey_productos.csv"
TX_PATH = BASE_DIR / "dataset_transacciones" / "hey_transacciones.csv"

for path in [CONV_PATH, CLIENTES_PATH, PRODUCTOS_PATH, TX_PATH]:
    print(path, "->", path.exists())

c:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\dataset_conversaciones\dataset_50k_anonymized.csv -> True
c:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\dataset_transacciones\hey_clientes.csv -> True
c:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\dataset_transacciones\hey_productos.csv -> True
c:\Users\carol\OneDrive\Imágenes\Escritorio\Datathon\dataset_transacciones\hey_transacciones.csv -> True


## 2. Carga de datos raw

In [4]:
conversaciones_raw = pd.read_csv(CONV_PATH)
clientes_raw = pd.read_csv(CLIENTES_PATH)
productos_raw = pd.read_csv(PRODUCTOS_PATH)
transacciones_raw = pd.read_csv(TX_PATH)

raw_shapes = pd.DataFrame([
    {"dataset": "conversaciones", "rows": conversaciones_raw.shape[0], "columns": conversaciones_raw.shape[1]},
    {"dataset": "clientes", "rows": clientes_raw.shape[0], "columns": clientes_raw.shape[1]},
    {"dataset": "productos", "rows": productos_raw.shape[0], "columns": productos_raw.shape[1]},
    {"dataset": "transacciones", "rows": transacciones_raw.shape[0], "columns": transacciones_raw.shape[1]},
])

raw_shapes

,dataset,rows,columns
0,conversaciones,49999,6
1,clientes,15025,22
2,productos,38909,13
3,transacciones,802384,22


## 3. Inspeccion inicial

Esta celda permite revisar columnas, tipos y primeras filas antes de limpiar.

In [8]:
datasets_raw = {
    "conversaciones": conversaciones_raw,
    "clientes": clientes_raw,
    "productos": productos_raw,
    "transacciones": transacciones_raw,
}

for name, df in datasets_raw.items():
    print("\n" + "=" * 80)
    print(name.upper(), df.shape)
    display(df.head(3))
    display(pd.DataFrame({"columna": df.columns, "dtype": [str(t) for t in df.dtypes]}))


CONVERSACIONES (49999, 6)


,input,output,date,conv_id,user_id,channel_source
0,"Me enteré de una promo ""Supercashback Pagos Gu...",Claro que puedo ayudarte! Para participar en l...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1
1,La tarjeta de crédito Hey Negocios es diferent...,Claro! La Tarjeta de Crédito Hey Negocios es d...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1
2,"Entiendo, gracias",¡De nada! Me alegra haber podido ayudarte. Si ...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1


,columna,dtype
0,input,object
1,output,object
2,date,object
3,conv_id,object
4,user_id,object
5,channel_source,int64



CLIENTES (15025, 22)


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0,False,True,es_MX,False,2,False
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0,False,True,es_MX,True,2,False
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,False,App,454,3,app_ios,8.0,False,True,es_MX,False,2,False


...

,producto_id,user_id,tipo_producto,fecha_apertura,estatus,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento,es_dato_sintetico
0,PRD-00000001,USR-00001,cuenta_debito,2023-06-26,activo,NaN,80954.60,NaN,NaN,NaN,NaN,2025-11-27,True
1,PRD-00000002,USR-00001,tarjeta_credito_hey,2022-10-16,activo,144000.0,88790.40,0.6166,35.71,NaN,NaN,2025-09-17,True
2,PRD-00000003,USR-00002,cuenta_debito,2025-04-03,activo,NaN,20712.54,NaN,NaN,NaN,NaN,2025-10-16,True


,columna,dtype
0,producto_id,object
1,user_id,object
2,tipo_producto,object
3,fecha_apertura,object
4,estatus,object
5,limite_credito,float64
6,saldo_actual,float64
7,utilizacion_pct,float64
8,tasa_interes_anual,float64
9,plazo_meses,float64



TRANSACCIONES (802384, 22)


,transaccion_id,user_id,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico,es_dato_sintetico
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.88,DivertidoPark,entretenimiento,"Nueva York, NY",completada,NaN,1,NaN,0.34,Cargo automático,14,Wednesday,True,app_ios,False,True
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.00,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,True
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.00,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,NaN,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,True


,columna,dtype
0,transaccion_id,object
1,user_id,object
2,producto_id,object
3,fecha_hora,object
4,tipo_operacion,object
5,canal,object
6,monto,float64
7,comercio_nombre,object
8,categoria_mcc,object
9,ciudad_transaccion,object


date tiene que ser convertido en fecha

## 4. Funciones reutilizables de limpieza y calidad

In [9]:
def normalize_text_columns(df):
    """Trim de textos y conversion de valores vacios a NA."""
    df = df.copy()
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "null": pd.NA})
        )
    return df


def convert_bool_columns(df, bool_cols):
    """Convierte columnas booleanas representadas como texto."""
    df = df.copy()
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "si": True,
        "sí": True,
        "no": False,
    }
    for col in bool_cols:
        if col in df.columns:
            if str(df[col].dtype) == "bool":
                df[col] = df[col].astype("boolean")
            else:
                df[col] = df[col].astype("string").str.strip().str.lower().map(mapping).astype("boolean")
    return df


def convert_numeric_columns(df, numeric_cols):
    df = df.copy()
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def missing_report(df):
    return (
        df.isna()
        .sum()
        .reset_index()
        .rename(columns={"index": "columna", 0: "nulos"})
        .assign(pct_nulos=lambda x: (x["nulos"] / len(df)).round(4))
        .sort_values(["pct_nulos", "nulos"], ascending=False)
    )


def unique_count_report(df):
    return pd.DataFrame({
        "columna": df.columns,
        "n_unicos": [df[col].nunique(dropna=True) for col in df.columns],
        "pct_unicos": [round(df[col].nunique(dropna=True) / len(df), 4) for col in df.columns],
    })

## 5. Limpieza de conversaciones

Fuente para detectar intencion, tema, frustracion, urgencia, producto mencionado, repeticion y canal.

### Eliminar duplicados y conversiones

In [ ]:
conversaciones = conversaciones_raw.copy()

# Limpieza ligera solo de columnas necesarias
cols_texto = ["channel_source", "input", "output"]

for col in cols_texto:
    conversaciones[col] = conversaciones[col].astype("string").str.strip()
    conversaciones.loc[
        conversaciones[col].str.lower().isin(["", "nan", "none", "null"]),
        col
    ] = pd.NA

# Transformación de fechas mixtas
date_series = conversaciones["date"].astype("string").str.strip()
date_series = date_series.mask(date_series.str.lower().isin(["", "nan", "none", "null", "nat"]))

conversaciones["date"] = pd.to_datetime(date_series, errors="coerce", format="mixed")

errores_fecha = conversaciones[conversaciones["date"].isna() & date_series.notna()]

print("Fechas inválidas:", len(errores_fecha))

if len(errores_fecha) > 0:
    display(errores_fecha[["date"]].head(10))

# Canal
conversaciones["channel_source"] = conversaciones["channel_source"].astype("string").str.strip()
conversaciones["canal_havi"] = conversaciones["channel_source"].map({"1": "texto", "2": "voz"}).fillna("desconocido")

# Texto limpio
conversaciones["input_clean"] = conversaciones["input"].fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
conversaciones["output_clean"] = conversaciones["output"].fillna("").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

# Duplicados por misma persona
cols_duplicados = ["user_id", "date", "channel_source", "input_clean", "output_clean"]

duplicados_mask = conversaciones.duplicated(
    subset=cols_duplicados,
    keep=False
)

print("Número de filas duplicadas:", duplicados_mask.sum())

duplicados = conversaciones[duplicados_mask].sort_values(
    by=["user_id", "date", "input_clean"]
)

display(duplicados.head(20))

conteo_duplicados = (
    conversaciones
    .groupby(cols_duplicados)
    .size()
    .reset_index(name="count")
    .query("count > 1")
    .sort_values(by="count", ascending=False)
)

display(conteo_duplicados.head(10))

# Eliminar duplicados dejando solo uno
filas_antes = conversaciones.shape[0]

conversaciones = conversaciones.drop_duplicates(
    subset=cols_duplicados,
    keep="first"
)

filas_despues = conversaciones.shape[0]

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)

print("Conversaciones limpias:", conversaciones.shape)
display(conversaciones.head())
display(missing_report(conversaciones))

Fechas inválidas: 0
Número de filas duplicadas: 169


,input,output,date,conv_id,user_id,channel_source,canal_havi,input_clean,output_clean
21181,Cómo puedo saber mi pin,Claro que puedo ayudarte! Para ver tu NIP en l...,2025-09-14,9F71A991-DE10-4EC5-A4A8-03EB8982D4E4,USR-00015,1,texto,cómo puedo saber mi pin,claro que puedo ayudarte! para ver tu nip en l...
45778,Cómo puedo saber mi pin,Claro que puedo ayudarte! Para ver tu NIP en l...,2025-09-14,03624752-A1ED-4C77-8D06-C0BFBDC84943,USR-00015,1,texto,cómo puedo saber mi pin,claro que puedo ayudarte! para ver tu nip en l...
5049,Atención a clientes,Lamento decirte que no tengo información acerc...,2025-08-30,288f1bd2-bbbb-4ac0-8f55-d624af61c91e,USR-00022,1,texto,atención a clientes,lamento decirte que no tengo información acerc...
5051,Atención a clientes,Lamento decirte que no tengo información acerc...,2025-08-30,288f1bd2-bbbb-4ac0-8f55-d624af61c91e,USR-00022,1,texto,atención a clientes,lamento decirte que no tengo información acerc...
26822,Nip bloqueado,Lamento escuchar eso. Si tu NIP está bloqueado...,2025-09-19,F197765F-90C3-49EB-B746-EE168A437E39,USR-00685,1,texto,nip bloqueado,lamento escuchar eso. si tu nip está bloqueado...
40668,Nip bloqueado,Lamento escuchar eso. Si tu NIP está bloqueado...,2025-09-19,EBE200D1-4EF8-479E-B65C-0259C6C5EBF5,USR-00685,1,texto,nip bloqueado,lamento escuchar eso. si tu nip está bloqueado...
29093,como obtengo una terminal,Claro que puedo ayudarte! Para obtener una Ter...,2025-08-26,b069b575-7514-4472-addf-8feb0b4aa0e2,USR-00690,1,texto,como obtengo una terminal,claro que puedo ayudarte! para obtener una ter...
48421,como obtengo una terminal,Claro que puedo ayudarte! Para obtener una Ter...,2025-08-26,76515298-9b8d-4c75-903b-b2c618168a73,USR-00690,1,texto,como obtengo una terminal,claro que puedo ayudarte! para obtener una ter...
39557,Mi tarjeta de crédito dice pendiente,"Lamento escuchar eso. El estatus ""Monto por pa...",2025-08-15,0DA37A28-13A4-4737-AAB0-486AB3C01B56,USR-00787,1,texto,mi tarjeta de crédito dice pendiente,"lamento escuchar eso. el estatus ""monto por pa..."
44440,Mi tarjeta de crédito dice pendiente,"Lamento escuchar eso. El estatus ""Monto por pa...",2025-08-15,DEF66E37-49C5-4B83-9F5E-CA4BD437FB20,USR-00787,1,texto,mi tarjeta de crédito dice pendiente,"lamento escuchar eso. el estatus ""monto por pa..."


,user_id,date,channel_source,input_clean,output_clean,count
8294,USR-02500,2025-09-10,1,atddhey194589,lamento decirte que no tengo información acerc...,3
10488,USR-03164,2025-09-10,1,cancelar tarjeta de crédito,claro! para cancelar tu tarjeta de crédito hey...,3
43501,USR-13103,2025-08-03,1,asesor,claro! si deseas hablar con un asesor de hey b...,3
25019,USR-07543,2025-09-15,1,hola,¡hola! ¿cómo estás? estoy aquí para ayudarte c...,3
16860,USR-05089,2025-08-14,1,havi,¡hola! claro que puedo ayudarte. ¿en qué puedo...,3
2300,USR-00690,2025-08-26,1,como obtengo una terminal,claro que puedo ayudarte! para obtener una ter...,2
78,USR-00022,2025-08-30,1,atención a clientes,lamento decirte que no tengo información acerc...,2
2286,USR-00685,2025-09-19,1,nip bloqueado,lamento escuchar eso. si tu nip está bloqueado...,2
2629,USR-00787,2025-08-15,1,mi tarjeta de crédito dice pendiente,"lamento escuchar eso. el estatus ""monto por pa...",2
2730,USR-00823,2025-10-02,1,como saber mi nip,claro que puedo ayudarte! para ver tu nip en l...,2


Filas antes: 49999
Filas después: 49912
Filas eliminadas: 87
Conversaciones limpias: (49912, 9)


,input,output,date,conv_id,user_id,channel_source,canal_havi,input_clean,output_clean
0,"Me enteré de una promo ""Supercashback Pagos Gu...",Claro que puedo ayudarte! Para participar en l...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,texto,"me enteré de una promo ""supercashback pagos gu...",claro que puedo ayudarte! para participar en l...
1,La tarjeta de crédito Hey Negocios es diferent...,Claro! La Tarjeta de Crédito Hey Negocios es d...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,texto,la tarjeta de crédito hey negocios es diferent...,claro! la tarjeta de crédito hey negocios es d...
2,"Entiendo, gracias",¡De nada! Me alegra haber podido ayudarte. Si ...,2025-08-17,000502c2-288c-41f6-b751-a8b45a376a81,USR-09092,1,texto,"entiendo, gracias",¡de nada! me alegra haber podido ayudarte. si ...
3,Hola,¡Hola! ¿Cómo estás? Estoy aquí para ayudarte c...,2025-09-16,00221607-cdb2-47f2-a9ac-913d4c7744c6,USR-09344,1,texto,hola,¡hola! ¿cómo estás? estoy aquí para ayudarte c...
4,Verificación de cuenta,Claro que puedo ayudarte con eso. Para validar...,2025-09-16,00221607-cdb2-47f2-a9ac-913d4c7744c6,USR-09344,1,texto,verificación de cuenta,claro que puedo ayudarte con eso. para validar...


,columna,nulos,pct_nulos
0,input,23,0.0005
1,output,0,0.0000
2,date,0,0.0000
3,conv_id,0,0.0000
4,user_id,0,0.0000
5,channel_source,0,0.0000
6,canal_havi,0,0.0000
7,input_clean,0,0.0000
8,output_clean,0,0.0000


## 6. Limpieza de clientes

Fuente de contexto demografico, relacion con Hey, satisfaccion y senales digitales.

In [29]:
clientes_raw.head()

,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0,False,True,es_MX,False,2,False
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0,False,True,es_MX,True,2,False
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,False,App,454,3,app_ios,8.0,False,True,es_MX,False,2,False
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,False,Fan Shop,837,16,app_ios,10.0,True,False,es_MX,True,3,False
4,USR-00005,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empresario,27000,816,True,False,Fan Shop,533,1,app_ios,7.0,False,True,es_MX,True,2,False


In [ ]:
clientes = clientes_raw.copy()
clientes = normalize_text_columns(clientes)

bool_cols_clientes = [
    "es_hey_pro",
    "nomina_domiciliada",
    "recibe_remesas",
    "usa_hey_shop",
    "tiene_seguro",
    "patron_uso_atipico",
]

numeric_cols_clientes = [
    "edad",
    "ingreso_mensual_mxn",
    "antiguedad_dias",
    "score_buro",
    "dias_desde_ultimo_login",
    "satisfaccion_1_10",
    "num_productos_activos",
]

clientes = convert_bool_columns(clientes, bool_cols_clientes)
clientes = convert_numeric_columns(clientes, numeric_cols_clientes)

clientes = clientes.drop_duplicates(subset=["user_id"], keep="first")

print("Clientes limpios:", clientes.shape)
display(clientes.head())
display(missing_report(clientes))

(15025, 22)
Clientes limpios: (15025, 22)


,user_id,edad,sexo,estado,ciudad,nivel_educativo,ocupacion,ingreso_mensual_mxn,antiguedad_dias,es_hey_pro,nomina_domiciliada,canal_apertura,score_buro,dias_desde_ultimo_login,preferencia_canal,satisfaccion_1_10,recibe_remesas,usa_hey_shop,idioma_preferido,tiene_seguro,num_productos_activos,patron_uso_atipico
0,USR-00001,21,M,Ciudad de México,CDMX - Benito Juárez,Preparatoria,Empleado,24500,1554,True,False,App,527,1,app_android,10.0,False,True,es_MX,False,2,False
1,USR-00002,18,M,Jalisco,Puerto Vallarta,Preparatoria,Estudiante,19000,1410,True,False,App,714,3,app_android,8.0,False,True,es_MX,True,2,False
2,USR-00003,23,H,Chihuahua,Cuauhtémoc,Licenciatura,Estudiante,14000,1174,True,False,App,454,3,app_ios,8.0,False,True,es_MX,False,2,False
3,USR-00004,32,SE,Nuevo León,Guadalupe,Posgrado,Empleado,61000,1168,False,False,Fan Shop,837,16,app_ios,10.0,True,False,es_MX,True,3,False
4,USR-00005,26,M,Ciudad de México,CDMX - Cuauhtémoc,Preparatoria,Empresario,27000,816,True,False,Fan Shop,533,1,app_ios,7.0,False,True,es_MX,True,2,False


,columna,nulos,pct_nulos
15,satisfaccion_1_10,751,0.0500
3,estado,432,0.0288
4,ciudad,432,0.0288
0,user_id,0,0.0000
1,edad,0,0.0000
2,sexo,0,0.0000
5,nivel_educativo,0,0.0000
6,ocupacion,0,0.0000
7,ingreso_mensual_mxn,0,0.0000
8,antiguedad_dias,0,0.0000


## 7. Limpieza de productos

Fuente para conocer portafolio, uso de credito, inversion, seguros y estado de productos.

In [31]:
productos = productos_raw.copy()
productos = normalize_text_columns(productos)

productos["fecha_apertura"] = pd.to_datetime(productos["fecha_apertura"], errors="coerce")
productos["fecha_ultimo_movimiento"] = pd.to_datetime(productos["fecha_ultimo_movimiento"], errors="coerce")

numeric_cols_productos = [
    "limite_credito",
    "saldo_actual",
    "utilizacion_pct",
    "tasa_interes_anual",
    "plazo_meses",
    "monto_mensualidad",
]

productos = convert_numeric_columns(productos, numeric_cols_productos)
productos = convert_bool_columns(productos, ["es_dato_sintetico"])

productos = productos.drop_duplicates(subset=["producto_id"], keep="first")

print("Productos limpios:", productos.shape)
display(productos.head())
display(missing_report(productos))

Productos limpios: (38909, 13)


,producto_id,user_id,tipo_producto,fecha_apertura,estatus,limite_credito,saldo_actual,utilizacion_pct,tasa_interes_anual,plazo_meses,monto_mensualidad,fecha_ultimo_movimiento,es_dato_sintetico
0,PRD-00000001,USR-00001,cuenta_debito,2023-06-26,activo,NaN,80954.60,NaN,NaN,NaN,NaN,2025-11-27,True
1,PRD-00000002,USR-00001,tarjeta_credito_hey,2022-10-16,activo,144000.0,88790.40,0.6166,35.71,NaN,NaN,2025-09-17,True
2,PRD-00000003,USR-00002,cuenta_debito,2025-04-03,activo,NaN,20712.54,NaN,NaN,NaN,NaN,2025-10-16,True
3,PRD-00000004,USR-00002,tarjeta_credito_hey,2025-07-18,activo,22000.0,6122.60,0.2783,34.08,NaN,NaN,2025-09-15,True
4,PRD-00000005,USR-00003,cuenta_debito,2023-03-26,activo,NaN,3454.65,NaN,NaN,NaN,NaN,2025-09-27,True


,columna,nulos,pct_nulos
9,plazo_meses,34359,0.8831
10,monto_mensualidad,34359,0.8831
5,limite_credito,24592,0.6320
7,utilizacion_pct,24592,0.6320
8,tasa_interes_anual,20118,0.5171
6,saldo_actual,3750,0.0964
0,producto_id,0,0.0000
1,user_id,0,0.0000
2,tipo_producto,0,0.0000
3,fecha_apertura,0,0.0000


## 8. Limpieza de transacciones

Fuente para comportamiento financiero, gasto, rechazos, disputas, recurrencia y patrones atipicos.

In [32]:
transacciones = transacciones_raw.copy()
transacciones = normalize_text_columns(transacciones)

transacciones["fecha_hora"] = pd.to_datetime(transacciones["fecha_hora"], errors="coerce")

numeric_cols_tx = [
    "monto",
    "intento_numero",
    "meses_diferidos",
    "cashback_generado",
    "hora_del_dia",
]

bool_cols_tx = [
    "es_internacional",
    "patron_uso_atipico",
    "es_dato_sintetico",
]

transacciones = convert_numeric_columns(transacciones, numeric_cols_tx)
transacciones = convert_bool_columns(transacciones, bool_cols_tx)
transacciones = transacciones.drop_duplicates(subset=["transaccion_id"], keep="first")

print("Transacciones limpias:", transacciones.shape)
display(transacciones.head())
display(missing_report(transacciones))

Transacciones limpias: (802213, 22)


,transaccion_id,user_id,producto_id,fecha_hora,tipo_operacion,canal,monto,comercio_nombre,categoria_mcc,ciudad_transaccion,estatus,motivo_no_procesada,intento_numero,meses_diferidos,cashback_generado,descripcion_libre,hora_del_dia,dia_semana,es_internacional,dispositivo,patron_uso_atipico,es_dato_sintetico
0,TXN-0000000055,USR-00001,PRD-00000002,2025-01-15 14:17:42,compra,app_ios,33.88,DivertidoPark,entretenimiento,"Nueva York, NY",completada,<NA>,1,NaN,0.34,Cargo automático,14,Wednesday,True,app_ios,False,True
1,TXN-0000000048,USR-00001,PRD-00000001,2025-01-17 00:31:56,cargo_recurrente,app_ios,249.00,GamerPass,servicios_digitales,CDMX - Benito Juárez,completada,<NA>,1,NaN,NaN,Cargo automático,0,Friday,False,app_ios,False,True
2,TXN-0000000018,USR-00001,PRD-00000001,2025-01-17 22:48:23,cargo_recurrente,app_huawei,399.00,CloudDrive MX,servicios_digitales,CDMX - Benito Juárez,completada,<NA>,1,NaN,NaN,inv hey,22,Friday,False,app_huawei,False,True
3,TXN-0000000043,USR-00001,PRD-00000001,2025-01-19 11:10:43,transf_salida,app_android,8910.00,<NA>,transferencia,CDMX - Benito Juárez,completada,<NA>,1,NaN,NaN,SPEI enviado,11,Sunday,False,app_android,False,True
4,TXN-0000000009,USR-00001,PRD-00000001,2025-02-15 07:03:50,compra,app_ios,568.59,QuickBite MX,restaurante,CDMX - Benito Juárez,completada,<NA>,1,NaN,5.69,dep. efvo,7,Saturday,False,app_ios,False,True


,columna,nulos,pct_nulos
13,meses_diferidos,784903,0.9784
11,motivo_no_procesada,775609,0.9668
14,cashback_generado,621607,0.7749
7,comercio_nombre,370451,0.4618
19,dispositivo,202013,0.2518
15,descripcion_libre,61741,0.0770
9,ciudad_transaccion,19264,0.0240
0,transaccion_id,0,0.0000
1,user_id,0,0.0000
2,producto_id,0,0.0000


## 9. Validacion de integridad referencial

Preguntas clave:

- Todos los `user_id` de conversaciones existen en clientes?
- Todos los `user_id` de productos existen en clientes?
- Todos los `user_id` de transacciones existen en clientes?
- Todos los `producto_id` de transacciones existen en productos?

In [33]:
clientes_ids = set(clientes["user_id"].dropna())
conv_ids = set(conversaciones["user_id"].dropna())
productos_user_ids = set(productos["user_id"].dropna())
tx_user_ids = set(transacciones["user_id"].dropna())

productos_ids = set(productos["producto_id"].dropna())
tx_producto_ids = set(transacciones["producto_id"].dropna())

integrity_checks = pd.DataFrame([
    {
        "check": "Usuarios en conversaciones no existen en clientes",
        "valor": len(conv_ids - clientes_ids),
    },
    {
        "check": "Usuarios en productos no existen en clientes",
        "valor": len(productos_user_ids - clientes_ids),
    },
    {
        "check": "Usuarios en transacciones no existen en clientes",
        "valor": len(tx_user_ids - clientes_ids),
    },
    {
        "check": "Productos en transacciones no existen en productos",
        "valor": len(tx_producto_ids - productos_ids),
    },
])

integrity_checks["status"] = np.where(integrity_checks["valor"].eq(0), "OK", "REVISAR")
integrity_checks

,check,valor,status
0,Usuarios en conversaciones no existen en clientes,0,OK
1,Usuarios en productos no existen en clientes,0,OK
2,Usuarios en transacciones no existen en clientes,0,OK
3,Productos en transacciones no existen en produ...,0,OK


## 10. Validaciones de calidad bancaria

rangos validos, duplicados, montos imposibles y fechas invalidas.

In [34]:
quality_checks = pd.DataFrame([
    {"check": "Clientes duplicados por user_id", "valor": int(clientes["user_id"].duplicated().sum())},
    {"check": "Productos duplicados por producto_id", "valor": int(productos["producto_id"].duplicated().sum())},
    {"check": "Transacciones duplicadas por transaccion_id", "valor": int(transacciones["transaccion_id"].duplicated().sum())},
    {"check": "Transacciones con monto nulo", "valor": int(transacciones["monto"].isna().sum())},
    {"check": "Transacciones con monto <= 0", "valor": int((transacciones["monto"] <= 0).sum())},
    {"check": "Clientes con edad fuera de 18-100", "valor": int(((clientes["edad"] < 18) | (clientes["edad"] > 100)).sum())},
    {"check": "Clientes con score_buro fuera de 295-850", "valor": int(((clientes["score_buro"] < 295) | (clientes["score_buro"] > 850)).sum())},
    {"check": "Clientes con satisfaccion fuera de 1-10", "valor": int(((clientes["satisfaccion_1_10"] < 1) | (clientes["satisfaccion_1_10"] > 10)).sum())},
    {"check": "Transacciones con fecha invalida", "valor": int(transacciones["fecha_hora"].isna().sum())},
    {"check": "Conversaciones con fecha invalida", "valor": int(conversaciones["date"].isna().sum())},
    {"check": "Conversaciones sin input", "valor": int(conversaciones["input"].isna().sum())},
    {"check": "Conversaciones sin output", "valor": int(conversaciones["output"].isna().sum())},
])

quality_checks["status"] = np.where(quality_checks["valor"].eq(0), "OK", "REVISAR")
quality_checks

,check,valor,status
0,Clientes duplicados por user_id,0,OK
1,Productos duplicados por producto_id,0,OK
2,Transacciones duplicadas por transaccion_id,0,OK
3,Transacciones con monto nulo,0,OK
4,Transacciones con monto <= 0,0,OK
5,Clientes con edad fuera de 18-100,0,OK
6,Clientes con score_buro fuera de 295-850,0,OK
7,Clientes con satisfaccion fuera de 1-10,0,OK
8,Transacciones con fecha invalida,0,OK
9,Conversaciones con fecha invalida,0,OK


## 11. Validaciones de fechas y categorias

Sirve para detectar problemas de consistencia temporal o categorias inesperadas antes de modelar.

In [35]:
date_ranges = pd.DataFrame([
    {
        "dataset": "conversaciones",
        "fecha_min": conversaciones["date"].min(),
        "fecha_max": conversaciones["date"].max(),
    },
    {
        "dataset": "productos_fecha_apertura",
        "fecha_min": productos["fecha_apertura"].min(),
        "fecha_max": productos["fecha_apertura"].max(),
    },
    {
        "dataset": "productos_fecha_ultimo_movimiento",
        "fecha_min": productos["fecha_ultimo_movimiento"].min(),
        "fecha_max": productos["fecha_ultimo_movimiento"].max(),
    },
    {
        "dataset": "transacciones",
        "fecha_min": transacciones["fecha_hora"].min(),
        "fecha_max": transacciones["fecha_hora"].max(),
    },
])

display(date_ranges)

category_columns = {
    "clientes.sexo": clientes["sexo"],
    "clientes.ocupacion": clientes["ocupacion"],
    "clientes.preferencia_canal": clientes["preferencia_canal"],
    "productos.tipo_producto": productos["tipo_producto"],
    "productos.estatus": productos["estatus"],
    "transacciones.tipo_operacion": transacciones["tipo_operacion"],
    "transacciones.estatus": transacciones["estatus"],
    "transacciones.categoria_mcc": transacciones["categoria_mcc"],
    "transacciones.canal": transacciones["canal"],
    "conversaciones.canal_havi": conversaciones["canal_havi"],
}

for name, series in category_columns.items():
    print("\n", name)
    display(series.value_counts(dropna=False).head(20).to_frame("count"))

,dataset,fecha_min,fecha_max
0,conversaciones,2025-01-07 03:16:22.067262795,2025-11-28 20:02:33.620581593
1,productos_fecha_apertura,2020-12-21 00:00:00.000000000,2025-11-27 00:00:00.000000000
2,productos_fecha_ultimo_movimiento,2025-08-30 00:00:00.000000000,2025-11-28 00:00:00.000000000
3,transacciones,2025-01-01 08:00:09.000000000,2025-12-15 13:59:36.000000000



 clientes.sexo


,count
sexo,
M,7281
H,7157
SE,587



 clientes.ocupacion


,count
ocupacion,
Empleado,8550
Independiente,3327
Empresario,1682
Estudiante,839
Desempleado,465
Jubilado,162


...

,count
categoria_mcc,
transferencia,366962
servicios_digitales,92542
restaurante,75483
supermercado,62992
gobierno,61386
entretenimiento,28955
ropa_accesorios,28222
tecnologia,17067
transporte,16674



 transacciones.canal


,count
canal,
app_ios,268258
app_android,226385
pos_fisico,110387
app_huawei,105557
cajero_banregio,26514
codi,26350
oxxo,23144
cajero_externo,12461
farmacia_ahorro,3157



 conversaciones.canal_havi


,count
canal_havi,
texto,46849
voz,3063


## 13. Resumen ejecutivo de validacion

Tabla unica para documentar la calidad inicial de datos.

In [39]:
validation_summary = pd.concat([integrity_checks, quality_checks], ignore_index=True)

extra_checks = pd.DataFrame([
    {
        "check": "Usuarios unicos en conversaciones",
        "valor": int(conversaciones["user_id"].nunique()),
        "status": "INFO",
    },
    {
        "check": "Usuarios unicos en clientes",
        "valor": int(clientes["user_id"].nunique()),
        "status": "INFO",
    },
    {
        "check": "Usuarios unicos en transacciones",
        "valor": int(transacciones["user_id"].nunique()),
        "status": "INFO",
    },
])

validation_summary = pd.concat([validation_summary, extra_checks], ignore_index=True)
validation_summary

,check,valor,status
0,Usuarios en conversaciones no existen en clientes,0,OK
1,Usuarios en productos no existen en clientes,0,OK
2,Usuarios en transacciones no existen en clientes,0,OK
3,Productos en transacciones no existen en produ...,0,OK
4,Clientes duplicados por user_id,0,OK
5,Productos duplicados por producto_id,0,OK
6,Transacciones duplicadas por transaccion_id,0,OK
7,Transacciones con monto nulo,0,OK
8,Transacciones con monto <= 0,0,OK
9,Clientes con edad fuera de 18-100,0,OK


## 14. Manejo estrategico de nulos para modelado

Estrategia:

- No se sobreescriben las tablas limpias originales.
- Se crean copias para modelado: `clientes_model`, `productos_model`, `transacciones_model`.
- Cada columna imputada conserva un flag `*_was_missing`.
- Los nulos estructurales se tratan como `no_aplica` o `0`, no como error.
- Los nulos informativos se imputan con mediana por grupo o categoria desconocida.

Esto evita perder informacion y permite que los modelos aprendan de la ausencia del dato.

### 14.0 Justificacion tecnica por variable

La estrategia de nulos se disena con criterio de produccion para datos financieros: distinguir entre **dato faltante real**, **dato no aplicable** y **dato sensible que no conviene inferir**. En vez de eliminar registros, se crean tablas de modelado y banderas `*_was_missing` para preservar trazabilidad.

| Variable | % Missing | Diagnostico | Accion recomendada | Justificacion | Impacto |
|---|---:|---|---|---|---|
| `clientes.satisfaccion_1_10` | 5.00% | Missing bajo, variable importante para churn/friccion. | Imputar mediana por `ocupacion`; fallback mediana global + `satisfaccion_1_10_was_missing`. | La satisfaccion puede variar por perfil laboral; mediana evita sesgo por outliers. | Mantiene variable util para riesgo de churn sin perder clientes. |
| `clientes.estado` | 2.88% | Missing bajo, categorica geografica sensible. | Categoria `No especificado` + `estado_was_missing`. | Inferir estado desde transacciones puede introducir sesgo o confundir residencia con lugar de compra. | Conserva cobertura y permite medir si falta de ubicacion tiene patron. |
| `clientes.ciudad` | 2.88% | Missing bajo, categorica granular. | Categoria `No especificado` + `ciudad_was_missing`. | Mejor no imputar ciudad con moda porque concentraria artificialmente usuarios en ciudades grandes. | Evita sesgo geografico en segmentacion y campanas. |
| `productos.plazo_meses` | 88.31% | Missing critico, pero estructural: solo aplica a creditos/prestamos. | Para prestamos: mediana por `tipo_producto`; no aplicables: `0` + flag. | El nulo no significa dato perdido en todos los casos, sino `no aplica`. | Util para calcular carga financiera sin penalizar productos no crediticios. |
| `productos.monto_mensualidad` | 88.31% | Missing critico, estructural. | Para prestamos: mediana por `tipo_producto`; no aplicables: `0` + flag. | Mensualidad solo tiene sentido en creditos con plazo. | Permite estimar presion financiera y capacidad de pago. |
| `productos.limite_credito` | 63.20% | Missing critico, estructural: solo aplica a productos de credito. | Para credito: mediana por `tipo_producto`; no credito: `0` + flag. | Imputar globalmente inflaria productos que no tienen limite. | Mejora features de capacidad crediticia y evita ruido en no-crediticios. |
| `productos.utilizacion_pct` | 63.20% | Missing critico, estructural. | Para credito: mediana por `tipo_producto`; no credito: `0` + flag. | La utilizacion solo aplica donde hay linea de credito. | Muy util para riesgo, estres financiero y recomendaciones de pago. |
| `productos.tasa_interes_anual` | 51.71% | Missing critico, parcialmente estructural. | Para productos con interes: mediana por `tipo_producto`; sin interes: `0` + flag. | No todos los productos generan interes, deuda o rendimiento. | Permite comparar costo/beneficio financiero sin crear tasas falsas. |
| `productos.saldo_actual` | 9.64% | Missing moderado, importante para liquidez/deuda. | Seguros: `0`; otros productos: mediana por `tipo_producto` + flag. | En seguros no hay saldo financiero comparable; en cuentas/creditos si conviene imputar. | Mantiene features de liquidez, deuda e inversion. |
| `transacciones.meses_diferidos` | 97.84% | Missing critico, pero estructural. | Imputar `0` + `meses_diferidos_was_missing`. | Nulo normalmente significa que la compra no fue a MSI. | Permite crear indicador `usa_msi` y medir financiamiento de consumo. |
| `transacciones.motivo_no_procesada` | 96.68% | Missing critico, estructural. | Si `estatus != no_procesada`: `no_aplica`; si rechazada y falta: `motivo_desconocido` + flag. | El motivo solo existe cuando la transaccion no se procesa. | Clave para detectar friccion, saldo insuficiente, limite excedido y riesgo operativo. |
| `transacciones.cashback_generado` | 77.49% | Missing critico, estructural. | Imputar `0` + flag. | Nulo significa que no genero cashback o no aplicaba beneficio. | Permite medir engagement con Hey Pro y valor economico para cliente. |
| `transacciones.comercio_nombre` | 46.18% | Missing critico, mixto. | Compras/cargos recurrentes sin comercio: `comercio_no_identificado`; otras operaciones: `no_aplica` + flag. | Transferencias/pagos no siempre tienen comercio; en compras si es senal de baja calidad. | Util para merchant analytics, recurrencias y deteccion de cargos sospechosos. |
| `transacciones.dispositivo` | 25.18% | Missing moderado, estructural por canal. | Si canal es app, imputar con `canal`; si no app: `no_aplica` + flag. | Dispositivo solo aplica naturalmente a canales digitales. | Ayuda a segmentar comportamiento digital y detectar actividad anomala. |
| `transacciones.descripcion_libre` | 7.70% | Missing bajo-moderado, texto. | Imputar `sin_descripcion` + flag. | No conviene inventar texto; se preserva ausencia como senal. | Util para NLP ligero, categorizacion y explicacion de movimientos. |
| `transacciones.ciudad_transaccion` | 2.40% | Missing bajo, geografica. | Si nacional: ciudad del cliente como fallback; si internacional: `internacional_no_especificada`; si no: `ciudad_no_especificada` + flag. | Imputar con ciudad del cliente solo es razonable para nacionales y debe marcarse. | Mejora deteccion de anomalias geograficas, con riesgo controlado. |

**Variables con missing critico (>40%)**

- `productos.plazo_meses`
- `productos.monto_mensualidad`
- `productos.limite_credito`
- `productos.utilizacion_pct`
- `productos.tasa_interes_anual`
- `transacciones.meses_diferidos`
- `transacciones.motivo_no_procesada`
- `transacciones.cashback_generado`
- `transacciones.comercio_nombre`

Aunque tienen missing alto, no se eliminan automaticamente porque en banca muchos nulos son **estructurales**: el dato no falta, simplemente no aplica al producto o tipo de operacion.

**Features derivadas recomendadas**

Crear missing flags para todas las variables imputadas, especialmente:

- `satisfaccion_1_10_was_missing`
- `estado_was_missing`
- `ciudad_was_missing`
- `limite_credito_was_missing`
- `utilizacion_pct_was_missing`
- `saldo_actual_was_missing`
- `motivo_no_procesada_was_missing`
- `comercio_nombre_was_missing`
- `dispositivo_was_missing`
- `ciudad_transaccion_was_missing`

**Riesgos de leakage o sesgo**

- `satisfaccion_1_10`: si se usa para predecir churn, puede ser proxy del target si la satisfaccion fue medida despues del evento. Usarla solo si temporalmente ocurre antes del evento modelado.
- `motivo_no_procesada`: no debe usarse para predecir si una transaccion sera rechazada si el motivo solo existe despues del rechazo. Si sirve para scoring historico del cliente o analisis posterior.
- `cashback_generado`: puede filtrar informacion de `es_hey_pro` y de compra completada. Util, pero con cuidado si el target es conversion a Hey Pro.
- `ciudad_transaccion`: imputar con ciudad del cliente puede esconder anomalias geograficas. Por eso se conserva `ciudad_transaccion_was_missing`.
- `monto_mensualidad`, `plazo_meses`, `utilizacion_pct`: no mezclar productos donde no aplican con creditos reales sin un flag de aplicabilidad.


In [ ]:
clientes_model = clientes.copy()
productos_model = productos.copy()
transacciones_model = transacciones.copy()

missing_targets = {
    "clientes": ["satisfaccion_1_10", "estado", "ciudad"],
    "productos": ["plazo_meses", "monto_mensualidad", "limite_credito", "utilizacion_pct", "tasa_interes_anual", "saldo_actual"],
    "transacciones": ["meses_diferidos", "motivo_no_procesada", "cashback_generado", "comercio_nombre", "dispositivo", "descripcion_libre", "ciudad_transaccion"],
}

def targeted_missing_report(df, cols, dataset_name):
    return pd.DataFrame([
        {
            "dataset": dataset_name,
            "columna": col,
            "nulos": int(df[col].isna().sum()),
            "pct_nulos": round(float(df[col].isna().mean()), 4),
        }
        for col in cols
        if col in df.columns
    ])

missing_before = pd.concat([
    targeted_missing_report(clientes_model, missing_targets["clientes"], "clientes"),
    targeted_missing_report(productos_model, missing_targets["productos"], "productos"),
    targeted_missing_report(transacciones_model, missing_targets["transacciones"], "transacciones"),
], ignore_index=True)

missing_before

### 14.1 Clientes

- `satisfaccion_1_10`: imputacion por mediana de `ocupacion`; fallback con mediana global.
- `estado` y `ciudad`: se marcan como `No especificado`, porque inferir residencia desde transacciones podria introducir sesgo.

In [ ]:
for col in ["satisfaccion_1_10", "estado", "ciudad"]:
    clientes_model[f"{col}_was_missing"] = clientes_model[col].isna()

satisfaccion_global_median = clientes_model["satisfaccion_1_10"].median()
satisfaccion_by_ocupacion = clientes_model.groupby("ocupacion")["satisfaccion_1_10"].transform("median")

clientes_model["satisfaccion_1_10"] = (
    clientes_model["satisfaccion_1_10"]
    .fillna(satisfaccion_by_ocupacion)
    .fillna(satisfaccion_global_median)
)

clientes_model["estado"] = clientes_model["estado"].fillna("No especificado")
clientes_model["ciudad"] = clientes_model["ciudad"].fillna("No especificado")

display(clientes_model[[
    "user_id",
    "ocupacion",
    "satisfaccion_1_10",
    "satisfaccion_1_10_was_missing",
    "estado",
    "estado_was_missing",
    "ciudad",
    "ciudad_was_missing",
]].head())

### 14.2 Productos

Muchos nulos son estructurales: no todos los productos tienen credito, tasa, plazo o mensualidad.

- Productos no crediticios: `limite_credito`, `utilizacion_pct`, `plazo_meses` y `monto_mensualidad` pasan a `0`.
- Productos crediticios con dato faltante: imputacion por mediana del `tipo_producto`.
- Productos sin interes: `tasa_interes_anual` pasa a `0`.
- `saldo_actual`: seguros se imputan como `0`; otros productos usan mediana por `tipo_producto`.

In [ ]:
product_missing_cols = ["plazo_meses", "monto_mensualidad", "limite_credito", "utilizacion_pct", "tasa_interes_anual", "saldo_actual"]

for col in product_missing_cols:
    productos_model[f"{col}_was_missing"] = productos_model[col].isna()

credit_products = [
    "tarjeta_credito_hey",
    "tarjeta_credito_garantizada",
    "tarjeta_credito_negocios",
    "credito_personal",
    "credito_auto",
    "credito_nomina",
]
loan_products = ["credito_personal", "credito_auto", "credito_nomina"]
interest_products = credit_products + ["inversion_hey"]
insurance_products = ["seguro_vida", "seguro_compras"]

is_credit = productos_model["tipo_producto"].isin(credit_products)
is_loan = productos_model["tipo_producto"].isin(loan_products)
is_interest = productos_model["tipo_producto"].isin(interest_products)
is_insurance = productos_model["tipo_producto"].isin(insurance_products)

def fill_numeric_by_product_type(df, col, applicable_mask, default_for_not_applicable=0):
    type_median = df.groupby("tipo_producto")[col].transform("median")
    global_median = df.loc[applicable_mask, col].median()
    df.loc[applicable_mask, col] = df.loc[applicable_mask, col].fillna(type_median).fillna(global_median)
    df.loc[~applicable_mask, col] = df.loc[~applicable_mask, col].fillna(default_for_not_applicable)
    return df

for col in ["limite_credito", "utilizacion_pct"]:
    productos_model = fill_numeric_by_product_type(productos_model, col, is_credit, 0)

for col in ["plazo_meses", "monto_mensualidad"]:
    productos_model = fill_numeric_by_product_type(productos_model, col, is_loan, 0)

productos_model = fill_numeric_by_product_type(productos_model, "tasa_interes_anual", is_interest, 0)

saldo_type_median = productos_model.groupby("tipo_producto")["saldo_actual"].transform("median")
saldo_global_median = productos_model.loc[~is_insurance, "saldo_actual"].median()
productos_model.loc[is_insurance, "saldo_actual"] = productos_model.loc[is_insurance, "saldo_actual"].fillna(0)
productos_model.loc[~is_insurance, "saldo_actual"] = (
    productos_model.loc[~is_insurance, "saldo_actual"]
    .fillna(saldo_type_median)
    .fillna(saldo_global_median)
)

display(productos_model[[
    "producto_id",
    "tipo_producto",
    "limite_credito",
    "utilizacion_pct",
    "tasa_interes_anual",
    "plazo_meses",
    "monto_mensualidad",
    "saldo_actual",
]].head(10))

### 14.3 Transacciones

- `meses_diferidos`: nulo significa sin MSI, se imputa `0`.
- `motivo_no_procesada`: `no_aplica` si la transaccion no fue rechazada; si fue rechazada y falta motivo, `motivo_desconocido`.
- `cashback_generado`: nulo significa cashback no generado, se imputa `0`.
- `comercio_nombre`: `no_aplica` para operaciones sin comercio; `comercio_no_identificado` para compras sin comercio.
- `dispositivo`: para canales app se usa el canal; para otros canales `no_aplica`.
- `descripcion_libre`: `sin_descripcion`.
- `ciudad_transaccion`: si falta y no es internacional, se usa la ciudad registrada del cliente; si no, categoria desconocida.

In [ ]:
tx_missing_cols = ["meses_diferidos", "motivo_no_procesada", "cashback_generado", "comercio_nombre", "dispositivo", "descripcion_libre", "ciudad_transaccion"]

for col in tx_missing_cols:
    transacciones_model[f"{col}_was_missing"] = transacciones_model[col].isna()

transacciones_model["meses_diferidos"] = transacciones_model["meses_diferidos"].fillna(0)
transacciones_model["cashback_generado"] = transacciones_model["cashback_generado"].fillna(0)

transacciones_model["motivo_no_procesada"] = np.where(
    transacciones_model["estatus"].eq("no_procesada") & transacciones_model["motivo_no_procesada"].isna(),
    "motivo_desconocido",
    transacciones_model["motivo_no_procesada"],
)
transacciones_model["motivo_no_procesada"] = transacciones_model["motivo_no_procesada"].fillna("no_aplica")

ops_with_merchant = ["compra", "cargo_recurrente"]
transacciones_model["comercio_nombre"] = np.where(
    transacciones_model["tipo_operacion"].isin(ops_with_merchant) & transacciones_model["comercio_nombre"].isna(),
    "comercio_no_identificado",
    transacciones_model["comercio_nombre"],
)
transacciones_model["comercio_nombre"] = transacciones_model["comercio_nombre"].fillna("no_aplica")

app_channels = ["app_ios", "app_android", "app_huawei"]
transacciones_model["dispositivo"] = np.where(
    transacciones_model["canal"].isin(app_channels) & transacciones_model["dispositivo"].isna(),
    transacciones_model["canal"],
    transacciones_model["dispositivo"],
)
transacciones_model["dispositivo"] = transacciones_model["dispositivo"].fillna("no_aplica")

transacciones_model["descripcion_libre"] = transacciones_model["descripcion_libre"].fillna("sin_descripcion")

cliente_city_map = clientes_model.set_index("user_id")["ciudad"].to_dict()
ciudad_cliente = transacciones_model["user_id"].map(cliente_city_map)

transacciones_model["ciudad_transaccion"] = np.where(
    transacciones_model["ciudad_transaccion"].isna() & transacciones_model["es_internacional"].eq(False),
    ciudad_cliente,
    transacciones_model["ciudad_transaccion"],
)
transacciones_model["ciudad_transaccion"] = np.where(
    transacciones_model["ciudad_transaccion"].isna() & transacciones_model["es_internacional"].eq(True),
    "internacional_no_especificada",
    transacciones_model["ciudad_transaccion"],
)
transacciones_model["ciudad_transaccion"] = transacciones_model["ciudad_transaccion"].fillna("ciudad_no_especificada")

display(transacciones_model[[
    "transaccion_id",
    "tipo_operacion",
    "estatus",
    "meses_diferidos",
    "motivo_no_procesada",
    "cashback_generado",
    "comercio_nombre",
    "dispositivo",
    "descripcion_libre",
    "ciudad_transaccion",
]].head(10))

### 14.4 Reporte despues de imputacion

Este reporte confirma que las columnas objetivo ya quedaron listas para feature engineering.

In [ ]:
missing_after = pd.concat([
    targeted_missing_report(clientes_model, missing_targets["clientes"], "clientes_model"),
    targeted_missing_report(productos_model, missing_targets["productos"], "productos_model"),
    targeted_missing_report(transacciones_model, missing_targets["transacciones"], "transacciones_model"),
], ignore_index=True)

missing_comparison = missing_before.merge(
    missing_after,
    on="columna",
    suffixes=("_before", "_after"),
)

missing_comparison = missing_comparison[[
    "columna",
    "dataset_before",
    "nulos_before",
    "pct_nulos_before",
    "dataset_after",
    "nulos_after",
    "pct_nulos_after",
]]

missing_comparison

## 15. Datos limpios listos para Customer 360

Al terminar esta fase quedan disponibles en memoria:

- `conversaciones`
- `clientes`
- `productos`
- `transacciones`
- `validation_summary`
- `clientes_model`
- `productos_model`
- `transacciones_model`

Siguiente paso: construir agregados por `user_id` para features conversacionales, transaccionales y de productos usando las tablas `*_model`.

## 16. Exportar outputs para Fase 2

Esta celda materializa el resultado de Fase 1. La Fase 2 debe consumir estos archivos y no repetir limpieza ni imputacion.

Outputs generados en `data_processed/`:

- `conversaciones_clean.pkl`
- `clientes_model.pkl`
- `productos_model.pkl`
- `transacciones_model.pkl`
- `validation_summary.pkl`
- `missing_comparison.pkl`


In [ ]:
PROCESSED_DIR = BASE_DIR / "data_processed"
PROCESSED_DIR.mkdir(exist_ok=True)

fase1_outputs = {
    "conversaciones_clean.pkl": conversaciones,
    "clientes_model.pkl": clientes_model,
    "productos_model.pkl": productos_model,
    "transacciones_model.pkl": transacciones_model,
    "validation_summary.pkl": validation_summary,
    "missing_comparison.pkl": missing_comparison,
}

for filename, df in fase1_outputs.items():
    output_path = PROCESSED_DIR / filename
    df.to_pickle(output_path)
    print(f"Guardado: {output_path} -> {df.shape}")

print("Fase 1 exportada correctamente. La Fase 2 puede iniciar desde data_processed/.")